# Model Optimizations 

In this notebook, the goal is to take our base models saved under the /work/models/ section and further enhance their predictability by fine-tuning their hyperparameters. 

## Importing our data 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, recall_score, precision_score, f1_score
from math import sqrt
import os
import csv

# Load the trainset
train_file_path = '/work/Downloaded/vbd-richard-bernat/trainn.csv'
test_file_path = '/work/Downloaded/test.csv'
train = pd.read_csv(train_file_path)
test = pd.read_csv(test_file_path)

# Placeholder for trainset preprocessing
#target = 'your_target_column'
#features = train.drop(target, axis=1)

from sklearn.preprocessing import LabelEncoder

# Encoding the target variable
label_encoder = LabelEncoder()
train['prognosis_encoded'] = label_encoder.fit_transform(train['prognosis'])

# Separating features and target variable
#X = train.drop(['id', 'prognosis', 'prognosis_encoded'], axis=1)
X = train.drop(['prognosis', 'prognosis_encoded'], axis=1)

y = train['prognosis_encoded']



# Split the trainset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=29)

#Importing our base models 

In [ ]:
import joblib  
from tensorflow.keras.models import load_model

ann =load_model('/work/models/ann.keras')
dt = joblib.load('/work/models/decision_tree.joblib')
rf = joblib.load('/work/models/random_forest.joblib')
xg_boost = joblib.load('/work/models/decision_tree.joblib')
knn = joblib.load('/work/models/decision_tree.joblib')


knn.fit(X_test,y_test)

2024-05-16 09:27:56.415136: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2024-05-16 09:27:56.415171: W tensorflow/stream_executor/cuda/cuda_driver.cc:263] failed call to cuInit: UNKNOWN ERROR (303)
2024-05-16 09:27:56.415192: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (p-82fc1969-7ff0-4344-a3e7-65ed95a291e5): /proc/driver/nvidia/version does not exist
2024-05-16 09:27:56.415439: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


DecisionTreeClassifier(max_depth=20)

## Hyperparameter tuning - ANN 

In [0]:
import numpy as np
from tensorflow.keras.models import Sequential, save_model
from tensorflow.keras.layers import Dense
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


# Define a function to create your model
def create_model(hidden_units=128, activation='relu', optimizer='adam'):
    model = Sequential([
        Dense(hidden_units, input_shape=(X_train.shape[1],), activation=activation),
        Dense(64, activation=activation),
        Dense(len(np.unique(y)), activation='softmax')  # Ensure output layer matches the number of classes
    ])
    model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

# Create KerasClassifier based on your model function
model = KerasClassifier(build_fn=create_model, epochs=100, batch_size=10, verbose=1)

# Define the grid search parameters
param_grid = {
    'hidden_units': [8, 16, 32,64,128],
    'activation': ['relu', 'tanh'],
    'optimizer': ['adam', 'rmsprop']
}

# Perform GridSearchCV
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, scoring='accuracy')
grid_result = grid.fit(X_train, y_train)

# Get the best hyperparameters
best_params = grid_result.best_params_

# Create a new model with the best hyperparameters
best_model = create_model(**best_params)

# Train the best model on the entire training dataset
best_model.fit(X_train, y_train, epochs=10, batch_size=10, verbose=1)

# Save the best model
save_model(best_model, '/work/models/best_ann_model.h5')

# Evaluate the best model on the test data
loss, accuracy = best_model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", accuracy)

Epoch 14/100
12/12 [==============================] - 0s 1ms/step - loss: 0.1484 - accuracy: 1.0000
Epoch 15/100
12/12 [==============================] - 0s 7ms/step - loss: 0.1277 - accuracy: 1.0000
Epoch 16/100
12/12 [==============================] - 0s 2ms/step - loss: 0.1111 - accuracy: 1.0000
Epoch 17/100
12/12 [==============================] - 0s 6ms/step - loss: 0.0982 - accuracy: 1.0000
Epoch 18/100
12/12 [==============================] - 0s 2ms/step - loss: 0.0882 - accuracy: 1.0000
Epoch 19/100
12/12 [==============================] - 0s 2ms/step - loss: 0.0786 - accuracy: 1.0000
Epoch 20/100
12/12 [==============================] - 0s 1ms/step - loss: 0.0716 - accuracy: 1.0000
Epoch 21/100
12/12 [==============================] - 0s 1ms/step - loss: 0.0645 - accuracy: 1.0000
Epoch 22/100
12/12 [==============================] - 0s 6ms/step - loss: 0.0586 - accuracy: 1.0000
Epoch 23/100
12/12 [==============================] - 0s 1ms/step - loss: 0.0538 - accuracy: 1.0000


In [ ]:
print(best_params)

{'activation': 'relu', 'hidden_units': 64, 'optimizer': 'adam'}


## Hyperparameter tuning - Decision Tree 

In [ ]:

dt_param_grid = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
dt_grid_search = GridSearchCV(dt, dt_param_grid, cv=5)
dt_grid_search.fit(X_train, y_train)
dt_best_params = dt_grid_search.best_params_

print("Best Hyperparameters for Decision Tree:")
for key, value in dt_best_params.items():
    print(key, ":", value)

best_dt_model = DecisionTreeClassifier(**dt_best_params)
joblib.dump(best_dt_model, '/work/models/best_dt_model.joblib')

Best Hyperparameters for Decision Tree:
max_depth : 30
min_samples_leaf : 1
min_samples_split : 5


['/work/models/best_dt_model.joblib']

In [ ]:
best_dt_model.fit(X_train,y_train)
accuracy = best_dt_model.score(X_test, y_test)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.6973684210526315


## Hyperparameter tuning - Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import joblib



# Define the grid search parameters
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Perform GridSearchCV
rf_grid_search = GridSearchCV(rf, rf_param_grid, cv=5)
rf_grid_search.fit(X_train, y_train)

# Get the best hyperparameters
rf_best_params = rf_grid_search.best_params_

# Print the best hyperparameters
print("Best Hyperparameters for Random Forest:")
for key, value in rf_best_params.items():
    print(key, ":", value)

# Create a new random forest model with the best hyperparameters
best_rf_model = RandomForestClassifier(**rf_best_params)

# Train the best model on the entire training dataset
best_rf_model.fit(X_train, y_train)  # Fit the model

# Save the best model
joblib.dump(best_rf_model, '/work/models/best_rf_model.joblib')

# Evaluate the best model on the test data
accuracy = best_rf_model.score(X_test, y_test)
print("Test Accuracy:", accuracy)


Best Hyperparameters for Random Forest:
max_depth : 10
min_samples_leaf : 1
min_samples_split : 10
n_estimators : 300
Test Accuracy: 0.9078947368421053


## Hyperparameter tuning - XG Boost  

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import joblib


# Define the XGBoost classifier
xgb = XGBClassifier()

# Define the grid search parameters
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

# Perform GridSearchCV
xgb_grid_search = GridSearchCV(xgb, xgb_param_grid, cv=5)
xgb_grid_search.fit(X_train, y_train)

# Get the best hyperparameters
xgb_best_params = xgb_grid_search.best_params_

# Print the best hyperparameters
print("Best Hyperparameters for XGBoost:")
for key, value in xgb_best_params.items():
    print(key, ":", value)

# Create a new XGBoost model with the best hyperparameters
best_xgb_model = XGBClassifier(**xgb_best_params)

# Train the best model on the entire training dataset
best_xgb_model.fit(X_train, y_train)  # Fit the model

# Save the best model
joblib.dump(best_xgb_model, '/work/models/best_xgb_model.joblib')

# Evaluate the best model on the test data
accuracy = best_xgb_model.score(X_test, y_test)
print("Test Accuracy:", accuracy)


Best Hyperparameters for XGBoost:
colsample_bytree : 0.8
learning_rate : 0.3
max_depth : 3
n_estimators : 200
subsample : 0.8
Test Accuracy: 0.9210526315789473


## Hyperparameter tuning - KNN Classifier 

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
best_accuracy = 0
best_k = 0
for k in range(1, 21):  # testing from 1 to 20 neighbors
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_k = k

print(f"Best K: {best_k} with Accuracy: {best_accuracy}")


joblib.dump( knn,'/work/models/best_knn_model.joblib')

Best K: 5 with Accuracy: 0.6973684210526315


['/work/models/best_knn_model.joblib']

## Working with LazyClassifier

In [ ]:
!pip install lazypredict

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 113.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Importing the data 
import pandas as pd
train=pd.read_csv('./Downloaded/vbd-richard-bernat/trainn.csv')
train.head()
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 65 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   sudden_fever           252 non-null    int64 
 1   headache               252 non-null    int64 
 2   mouth_bleed            252 non-null    int64 
 3   nose_bleed             252 non-null    int64 
 4   muscle_pain            252 non-null    int64 
 5   joint_pain             252 non-null    int64 
 6   vomiting               252 non-null    int64 
 7   rash                   252 non-null    int64 
 8   diarrhea               252 non-null    int64 
 9   hypotension            252 non-null    int64 
 10  pleural_effusion       252 non-null    int64 
 11  ascites                252 non-null    int64 
 12  gastro_bleeding        252 non-null    int64 
 13  swelling               252 non-null    int64 
 14  nausea                 252 non-null    int64 
 15  chills                 

In [ ]:
test=pd.read_csv('./Downloaded/vbd-richard-bernat/testt.csv')
test.head()
# Encoding the target variable
# label_encoder = LabelEncoder()
# train['prognosis_encoded'] = label_encoder.fit_transform(train['prognosis'])


,sudden_fever,headache,mouth_bleed,nose_bleed,muscle_pain,joint_pain,vomiting,rash,diarrhea,hypotension,...,breathing_restriction,toe_inflammation,finger_inflammation,lips_irritation,itchiness,ulcers,toenail_loss,speech_problem,bullseye_rash,prognosis
0,1,0,0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,Chikungunya
1,1,0,0,0,1,1,1,1,0,1,...,0,0,0,0,0,0,1,0,0,Dengue
2,1,1,1,1,0,1,0,1,0,1,...,0,1,0,1,0,0,0,0,0,Rift Valley fever
3,1,1,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Yellow Fever
4,0,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Zika


In [ ]:
train.info(),test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 65 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   sudden_fever           252 non-null    int64 
 1   headache               252 non-null    int64 
 2   mouth_bleed            252 non-null    int64 
 3   nose_bleed             252 non-null    int64 
 4   muscle_pain            252 non-null    int64 
 5   joint_pain             252 non-null    int64 
 6   vomiting               252 non-null    int64 
 7   rash                   252 non-null    int64 
 8   diarrhea               252 non-null    int64 
 9   hypotension            252 non-null    int64 
 10  pleural_effusion       252 non-null    int64 
 11  ascites                252 non-null    int64 
 12  gastro_bleeding        252 non-null    int64 
 13  swelling               252 non-null    int64 
 14  nausea                 252 non-null    int64 
 15  chills                 

(None, None)

In [ ]:
X,y = train.drop(['prognosis'],axis=1), train.prognosis
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3)

In [ ]:
from lazypredict.Supervised import LazyClassifier
clf = LazyClassifier(verbose=0,predictions=True)
models,predictions = clf.fit(X_train,X_test,y_train,y_test)
models

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
NuSVC,0.92,0.92,None,0.92,0.02
BernoulliNB,0.91,0.91,None,0.91,0.01
NearestCentroid,0.91,0.91,None,0.91,0.02
ExtraTreesClassifier,0.91,0.91,None,0.91,0.13
SVC,0.91,0.91,None,0.91,0.02
LogisticRegression,0.88,0.89,None,0.88,0.03
RandomForestClassifier,0.88,0.88,None,0.88,0.12
LinearDiscriminantAnalysis,0.87,0.87,None,0.86,0.09
SGDClassifier,0.86,0.87,None,0.85,0.07


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=82fc1969-7ff0-4344-a3e7-65ed95a291e5' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>